# Название проекта

## Описание проекта

## Цель исследования

## Ход исследования

В процессе исследования будут проведены следующие действия
- подготовка данных: загрузка и изучение общей информации из предоставленных датасетов
- предобработка данных: обработка пропущенных значений, обработка дублей, корректировка данных
- проведение анализа представленных данных: выявление закономерностей и аномалий, распределения данных по параметрам
- объединение данных в единый датасет и его расширение дополнительной информацией на основе представленных данных
- выявление лучших моделей для задач регрессии и бинарной классификации целевых признаков с использованием пайплайнов
- резюмирование полученных результатов, формулировка общих выводов и рекомендаций

## Настройка окружения

### Импорты

In [17]:
# %pip install -Uq scikit-learn
# %pip install phik -q
# %pip install shap -q

# imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import shap

# import warnings
# warnings.filterwarnings('ignore', category=pd.core.common.SettingWithCopyWarning)

from phik import phik_matrix
from plotly.subplots import make_subplots
from scipy import stats as st
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_auc_score,
    f1_score,
    make_scorer,
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    LabelEncoder,
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
    PolynomialFeatures,
)

from sklearn.model_selection import (
    train_test_split, 
    GridSearchCV, 
    RandomizedSearchCV,
)

# загружаем класс для работы с пропусками
from sklearn.impute import SimpleImputer, KNNImputer

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC


### Настройки отображения

In [18]:
# output settings
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option('display.max_colwidth', None)

### Объявление функций

In [19]:
def check_size(current_df, original_df):
    print("Количество записей: {}".format(len(current_df)))
    print("Процент от начального объема данных: {:.2%}".format(len(current_df) / len(original_df)))

In [20]:
import os

HOST = "https://code.s3.yandex.net"
HTTP_PREFIX = "http"


# load csv
def load_csv(dataset_path: str, **kwargs):
    # def load_csv(dataset_path, delimiter=",", decimal=".", index_col=None):
    # check server request --> relative path --> absolute path --> yandex server request
    path = (
        dataset_path
        if dataset_path.startswith(HTTP_PREFIX)
        else "." + dataset_path if os.path.exists("." + dataset_path)
        else dataset_path if os.path.exists(dataset_path) 
        else HOST + dataset_path
    )
    print("Dataset path:", path)
    try:
        return pd.read_csv(filepath_or_buffer=path, **kwargs)
        # return pd.read_csv(path, delimiter=delimiter, decimal=decimal, index_col=index_col)
    except Exception as ex:
        print("Could not load csv. Exception:", str(ex))

In [ ]:
def df_init_analysis(df_name: str, df: pd.DataFrame):
    print("=" * 50)
    print(f"{df_name}\n")
    print("Общая информация\n")
    print(df.info())

    print("\nБазовая статистика по данным")
    display(df.describe(include='all'))

    print("\nИнформация по колонкам\n")
    df_init_analysis_column_names = [
        "column_name",
        "type",
        "na_count",
        "empty_count",
        "unique_count",
    ]

    df_init_analysis_data = []
    for column_name in df.columns.tolist():
        df_init_column_data = []
        df_init_column_data.append(column_name)
        df_init_column_data.append(df[column_name].dtype)
        df_init_column_data.append(df[column_name].isna().sum())
        df_init_column_data.append(sum(df[column_name] == ""))
        df_init_column_data.append(df[column_name].nunique())
        df_init_analysis_data.append(df_init_column_data)

    df_init = pd.DataFrame(columns=df_init_analysis_column_names, data=df_init_analysis_data)
    display(df_init)

    print(f"\nКоличество явных дубликатов: {df.duplicated().sum()}\n")

In [30]:
def rename_columns_to_lowercase(columns_list):
    """
    Преобразует названия столбцов датафрейма в lowercase и заменяет пробелы на "_".
    
    :param columns_list: Список текущих названий столбцов датафрейма
    :return: Отформатированный список новых названий столбцов
    """
    new_columns = []
    for col in columns_list:
        # Приводим название столбца к нижнему регистру и заменяем пробелы на _
        new_col_name = col.lower().replace(' ', '_')
        new_columns.append(new_col_name)
    return new_columns

## Загрузка данных

In [22]:
df_heart_train_original = load_csv(".private/heart_train.csv", index_col=0)
df_heart_train = df_heart_train_original.copy().set_index("id")
df_heart_train.head()

Dataset path: .private/heart_train.csv


,Age,Cholesterol,Heart rate,Diabetes,Family History,Smoking,Obesity,Alcohol Consumption,Exercise Hours Per Week,Diet,Previous Heart Problems,Medication Use,Stress Level,Sedentary Hours Per Day,Income,BMI,Triglycerides,Physical Activity Days Per Week,Sleep Hours Per Day,Heart Attack Risk (Binary),Blood sugar,CK-MB,Troponin,Gender,Systolic blood pressure,Diastolic blood pressure
id,,,,,,,,,,,,,,,,,,,,,,,,,,
2664,0.36,0.73,0.07,1.00,1.00,1.00,1.00,1.00,0.54,1,1.00,0.00,8.00,0.23,0.11,0.46,0.98,3.00,0.33,0.00,0.23,0.05,0.04,Male,0.21,0.71
9287,0.20,0.33,0.05,1.00,1.00,0.00,0.00,1.00,0.07,2,1.00,0.00,9.00,0.29,0.16,0.12,0.52,3.00,0.83,0.00,0.15,0.02,0.00,Female,0.41,0.57
5379,0.61,0.86,0.06,1.00,0.00,1.00,1.00,1.00,0.94,2,1.00,1.00,6.00,0.55,0.60,0.37,0.01,2.00,1.00,0.00,0.23,0.05,0.04,Female,0.24,0.22
8222,0.73,0.01,0.05,0.00,0.00,1.00,0.00,1.00,0.70,0,0.00,1.00,3.00,0.33,0.08,0.05,0.13,0.00,0.33,1.00,0.23,0.05,0.04,Female,0.35,0.27
4047,0.78,0.76,0.02,0.00,0.00,1.00,0.00,1.00,0.41,1,0.00,0.00,8.00,0.52,0.34,0.83,0.07,5.00,1.00,1.00,0.23,0.05,0.04,Male,0.62,0.44


In [23]:
df_heart_test_original = load_csv(".private/heart_test.csv", index_col=0)
df_heart_test = df_heart_test_original.copy().set_index("id")
df_heart_test.head()

Dataset path: .private/heart_test.csv


,Age,Cholesterol,Heart rate,Diabetes,Family History,Smoking,Obesity,Alcohol Consumption,Exercise Hours Per Week,Diet,Previous Heart Problems,Medication Use,Stress Level,Sedentary Hours Per Day,Income,BMI,Triglycerides,Physical Activity Days Per Week,Sleep Hours Per Day,Blood sugar,CK-MB,Troponin,Gender,Systolic blood pressure,Diastolic blood pressure
id,,,,,,,,,,,,,,,,,,,,,,,,,
7746,0.49,0.26,0.06,0.00,1.00,1.00,1.00,1.00,0.36,2,0.00,0.00,8.00,0.19,0.59,0.28,0.31,1.00,0.33,0.23,0.05,0.04,Male,0.28,0.37
4202,0.22,0.95,0.08,1.00,0.00,0.00,1.00,0.00,1.00,2,1.00,1.00,5.00,0.33,0.60,0.47,0.09,0.00,0.17,0.23,0.05,0.04,Female,0.70,0.44
6632,0.63,0.09,0.06,0.00,1.00,1.00,1.00,0.00,1.00,0,0.00,0.00,10.00,0.78,0.37,0.41,0.21,7.00,1.00,0.10,0.00,0.09,Male,0.46,0.78
4639,0.46,0.57,0.06,1.00,1.00,1.00,1.00,1.00,0.44,0,0.00,0.00,10.00,0.79,0.37,0.91,0.16,0.00,0.67,0.20,0.06,0.27,Female,0.74,0.26
4825,0.72,0.49,0.02,1.00,0.00,1.00,0.00,1.00,0.51,0,0.00,0.00,7.00,0.07,0.73,0.76,0.58,5.00,0.00,0.23,0.05,0.04,Male,0.41,0.40


Данные загружены. 

Выглядят уже частично предобработанными и нормализованными.

## Предобработка данных

### Первичный анализ данных

In [24]:
df_dict = {
    "df_heart_train": df_heart_train,
}

In [25]:
for df_name, df in df_dict.items():
    df_init_analysis(df_name, df)

df_heart_train

Общая информация

<class 'pandas.core.frame.DataFrame'>
Index: 8685 entries, 2664 to 7270
Data columns (total 26 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Age                              8685 non-null   float64
 1   Cholesterol                      8685 non-null   float64
 2   Heart rate                       8685 non-null   float64
 3   Diabetes                         8442 non-null   float64
 4   Family History                   8442 non-null   float64
 5   Smoking                          8442 non-null   float64
 6   Obesity                          8442 non-null   float64
 7   Alcohol Consumption              8442 non-null   float64
 8   Exercise Hours Per Week          8685 non-null   float64
 9   Diet                             8685 non-null   int64  
 10  Previous Heart Problems          8442 non-null   float64
 11  Medication Use                   8442 non-null   f

,Age,Cholesterol,Heart rate,Diabetes,Family History,Smoking,Obesity,Alcohol Consumption,Exercise Hours Per Week,Diet,Previous Heart Problems,Medication Use,Stress Level,Sedentary Hours Per Day,Income,BMI,Triglycerides,Physical Activity Days Per Week,Sleep Hours Per Day,Heart Attack Risk (Binary),Blood sugar,CK-MB,Troponin,Gender,Systolic blood pressure,Diastolic blood pressure
count,"8,685.00","8,685.00","8,685.00","8,442.00","8,442.00","8,442.00","8,442.00","8,442.00","8,685.00","8,685.00","8,442.00","8,442.00","8,442.00","8,685.00","8,685.00","8,685.00","8,685.00","8,442.00","8,685.00","8,685.00","8,685.00","8,685.00","8,685.00",8685,"8,685.00","8,685.00"
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Male,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5882,NaN,NaN
mean,0.45,0.50,0.05,0.65,0.49,0.90,0.50,0.60,0.50,1.06,0.50,0.50,5.49,0.50,0.50,0.50,0.51,3.51,0.50,0.35,0.23,0.05,0.04,NaN,0.45,0.50
std,0.23,0.28,0.02,0.48,0.50,0.30,0.50,0.49,0.28,0.87,0.50,0.50,2.87,0.29,0.28,0.28,0.29,2.28,0.33,0.48,0.08,0.08,0.06,NaN,0.17,0.17
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,NaN,0.00,0.00
25%,0.26,0.27,0.03,0.00,0.00,1.00,0.00,0.00,0.26,0.00,0.00,0.00,3.00,0.26,0.25,0.25,0.26,2.00,0.17,0.00,0.23,0.05,0.04,NaN,0.30,0.35
50%,0.46,0.50,0.05,1.00,0.00,1.00,0.00,1.00,0.50,1.00,0.00,1.00,5.00,0.50,0.49,0.49,0.50,3.00,0.50,0.00,0.23,0.05,0.04,NaN,0.45,0.50
75%,0.64,0.75,0.07,1.00,1.00,1.00,1.00,1.00,0.75,2.00,1.00,1.00,8.00,0.74,0.74,0.74,0.75,6.00,0.83,1.00,0.23,0.05,0.04,NaN,0.60,0.65



Информация по колонкам



,column_name,type,na_count,empty_count,unique_count
0,Age,float64,0,0,77
1,Cholesterol,float64,0,0,282
2,Heart rate,float64,0,0,87
3,Diabetes,float64,243,0,2
4,Family History,float64,243,0,2
5,Smoking,float64,243,0,2
6,Obesity,float64,243,0,2
7,Alcohol Consumption,float64,243,0,2
8,Exercise Hours Per Week,float64,0,0,7933
9,Diet,int64,0,0,4



Количество дубликатов: 0



Тренировочные данные содержат 8685 записей. Целевой признак -- `Heart Attack Risk (Binary)`. Количество явных дубликатов равно 0.

Данные нормализованы. За редким исключением значения лежат в диапазоне от 0 до 1.

Наблюдаются пропущенные значения для некоторых признаков:
- Diabetes
- Family History
- Smoking
- Obesity
- Alcohol Consumption
- Previous Heart Problems
- Medication Use
- Stress Level
- Physical Activity Days Per Week

Присутствуют количественные и категориальные признаки.

Количественные признаки:
- Age
- Cholesterol
- Heart rate
- Exercise Hours Per Week
- Sedentary Hours Per Day
- Income
- BMI
- Triglycerides
- Blood sugar
- CK-MB
- Troponin
- Systolic blood pressure
- Diastolic blood pressure

Категориальные признаки:
- Diabetes
- Family History
- Smoking
- Obesity
- Alcohol Consumption
- Diet
- Previous Heart Problems
- Medication Use
- Stress Level
- Physical Activity Days Per Week
- Sleep Hours Per Day
- Gender

Интересно, что тип признака Gender определен как object и содержит 4 уникальных значения.

In [26]:
df_heart_train.Gender.unique()

array(['Male', 'Female', '1.0', '0.0'], dtype=object)

Видимо, ошибки при получении данных.

```text
Международно признанная цифровая классификация пола основывается на стандарте ISO/IEC 5218 («Representation of human sexes»). Согласно данному международному стандарту, приняты следующие цифровые обозначения:

0 — неизвестный пол (Not known)
1 — мужской пол (Male)
2 — женский пол (Female)
9 — неприменимо (Inapplicable)
```

Также стоит отметить, что почти все категориальные признаки имеют тип float. Выглядит логичным привести категориальыне признаки к типу category.

### Переименование столбцов

In [27]:
df_heart_train.rename(columns=lambda x: x.lower().replace(" ", "_"), inplace=True)
df_heart_train.columns.to_list()

['age',
 'cholesterol',
 'heart_rate',
 'diabetes',
 'family_history',
 'smoking',
 'obesity',
 'alcohol_consumption',
 'exercise_hours_per_week',
 'diet',
 'previous_heart_problems',
 'medication_use',
 'stress_level',
 'sedentary_hours_per_day',
 'income',
 'bmi',
 'triglycerides',
 'physical_activity_days_per_week',
 'sleep_hours_per_day',
 'heart_attack_risk_(binary)',
 'blood_sugar',
 'ck-mb',
 'troponin',
 'gender',
 'systolic_blood_pressure',
 'diastolic_blood_pressure']

Дополнительно переименуем колонку с целевым признаком.

In [28]:
df_heart_train.rename(columns={'heart_attack_risk_(binary)': 'heart_attack_risk'}, inplace=True)
df_heart_train.columns.to_list()

['age',
 'cholesterol',
 'heart_rate',
 'diabetes',
 'family_history',
 'smoking',
 'obesity',
 'alcohol_consumption',
 'exercise_hours_per_week',
 'diet',
 'previous_heart_problems',
 'medication_use',
 'stress_level',
 'sedentary_hours_per_day',
 'income',
 'bmi',
 'triglycerides',
 'physical_activity_days_per_week',
 'sleep_hours_per_day',
 'heart_attack_risk',
 'blood_sugar',
 'ck-mb',
 'troponin',
 'gender',
 'systolic_blood_pressure',
 'diastolic_blood_pressure']

Аналогично для тестовых данных.

In [29]:
df_heart_test.rename(columns=lambda x: x.lower().replace(" ", "_"), inplace=True)
df_heart_test.columns.to_list()

['age',
 'cholesterol',
 'heart_rate',
 'diabetes',
 'family_history',
 'smoking',
 'obesity',
 'alcohol_consumption',
 'exercise_hours_per_week',
 'diet',
 'previous_heart_problems',
 'medication_use',
 'stress_level',
 'sedentary_hours_per_day',
 'income',
 'bmi',
 'triglycerides',
 'physical_activity_days_per_week',
 'sleep_hours_per_day',
 'blood_sugar',
 'ck-mb',
 'troponin',
 'gender',
 'systolic_blood_pressure',
 'diastolic_blood_pressure']

### Обработка пропусков

Получим список признаков с пропущенными значениями

In [31]:
na_cols_original_names = [
    "Diabetes",
    "Family History",
    "Smoking",
    "Obesity",
    "Alcohol Consumption",
    "Previous Heart Problems",
    "Medication Use",
    "Stress Level",
    "Physical Activity Days Per Week",
]
na_cols = rename_columns_to_lowercase(na_cols_original_names)
na_cols

['diabetes',
 'family_history',
 'smoking',
 'obesity',
 'alcohol_consumption',
 'previous_heart_problems',
 'medication_use',
 'stress_level',
 'physical_activity_days_per_week']

Будем заполнять пропущенные значения с помощью пайплайна и трансформера. Непосредственно заполнение пропусков будет произведено на этапе обучения моделей с использованием пайплайнов. На данном этапе просто объявим инструменты для заполнения пропусков.

In [32]:
missing_values_pipe = Pipeline(
    [
        ("imputer", SimpleImputer(missing_values=np.nan, strategy="most_frequent")),
    ],
)

na_transformer = ColumnTransformer(
    transformers=[
        ("na", missing_values_pipe, na_cols),
    ],
    remainder="passthrough",
)

### Преобразование типов данных

Преобразуем категориальные признаки в тип данных `category`. Предварительно приведем их к целочисленному типу.

In [33]:
cat_cols_original_names = [
    "Diabetes",
    "Family History",
    "Smoking",
    "Obesity",
    "Alcohol Consumption",
    "Diet",
    "Previous Heart Problems",
    "Medication Use",
    "Stress Level",
    "Physical Activity Days Per Week",
    "Sleep Hours Per Day",
    "Gender",
]
cat_cols = rename_columns_to_lowercase(cat_cols_original_names)
cat_cols

['diabetes',
 'family_history',
 'smoking',
 'obesity',
 'alcohol_consumption',
 'diet',
 'previous_heart_problems',
 'medication_use',
 'stress_level',
 'physical_activity_days_per_week',
 'sleep_hours_per_day',
 'gender']

А не можем мы привести их к типу int по причине наличия пропусков значений в этих столбцах : (

### Наличие дубликатов

Проверим наличие явных дубликатов

In [34]:
def print_df_duplicates(df_name, df):
    # Создаем маску для строк с дубликатами
    mask = df.duplicated()
    if mask.sum() > 0:
        # Выводим дубликаты
        print("=" * 50)
        print(f"\n{df_name}: total: {mask.sum()}\n")
        display(df[mask].head())
    return mask.sum()

In [35]:
duplicates_count = 0
for df_name, df in df_dict.items():
    duplicates_count += print_df_duplicates(df_name, df)
if duplicates_count == 0:
    print("Явных дубликтов не обнаружено")

Явных дубликтов не обнаружено


### Ошибки в числовых значениях

In [36]:
negative_values_flag = False
for df_name, df in df_dict.items():
    for df_column_name in list(df.select_dtypes(include="number").columns):
        if df[df_column_name][df[df_column_name] < 0].count() > 0:
            print(f'Количество отрицательных значений в {df_name}.{df_column_name}: {df[df_column_name][df[df_column_name] < 0].count()}')
            negative_values_flag = True
if not negative_values_flag:
    print("Отрицательные значения отсутствуют")

Отрицательные значения отсутствуют


### Промежуточные выводы

В ходе предварительной обработки данных было сделано следующее:
- проведена проверка на наличие пропусков в данных, пропуски обнаружены, будут обработаны с использованием пайплайнов
- проведена проверка на наличие явных и неявных дубликатов, ошибки в категориальных данных устранены, строки из пробелов заменены на значение `nan`
- проведена проверка на наличие неявных ошибок в числовых данных, ошибок не выявлено

Дальнейшие шаги будем проводить в разрезе каждой из задач

## Исследовательский анализ данных

### Количественные признаки

In [ ]:
# количество корзин в зависимости от количества уникальных значений
def get_bins_count(value):
    if value > 100:
        return value // 10
    return value

Создадим необходимые функции для отображения графиков

In [ ]:
# histogram
def get_histogram(df, df_column_name, df_target_column_name):
    fig = px.histogram(
        df, 
        x=df_column_name, 
        color=df_target_column_name,
        color_discrete_map={
            'неудовлетворен': 'Coral',
            'нейтрально': 'lightyellow',
            'удовлетворен': 'SeaGreen',
        },
        nbins=get_bins_count(df[df_column_name].nunique())
    )

    fig.update_layout(
        title_text=f"Гистограмма распределения {df_column_name}",
        xaxis_title_text=f"{df_column_name}",
        yaxis_title_text="Кол-во",
        barmode='group',
        legend_title=None,
    )

    fig.show()

In [ ]:
# boxplot
def get_boxplot(df, df_column_name, df_target_column_name):
    fig = go.Figure()

    fig.add_trace(
        go.Box(
            x=df[df_target_column_name == "неудовлетворен"][
                df_column_name
            ],
            name="неудовлетворен",
            line=dict(color="red"),
            fillcolor="Coral",
        )
    )

    fig.add_trace(
        go.Box(
            x=df[df_target_column_name == "нейтрально"][
                df_column_name
            ],
            name="нейтрально",
            line=dict(color="yellow"),
            fillcolor="lightyellow",
        )
    )

    fig.add_trace(
        go.Box(
            x=df[df_target_column_name == "удовлетворен"][
                df_column_name
            ],
            name="удовлетворен",
            line=dict(color="green"),
            fillcolor="SeaGreen",
        )
    )

    fig.update_layout(
        title_text=f"Распределение значений {df_column_name}",
        xaxis_title_text=f"{df_column_name}",
        legend_title=None,
    )

    fig.show()

In [ ]:
def get_histogram_boxplot(df, df_column_name, df_target_column_name):
    get_histogram(df, df_column_name, df_target_column_name)
    get_boxplot(df, df_column_name, df_target_column_name)

In [ ]:
for df_column_name in df.select_dtypes(include="number").drop(["index"], axis=1).columns:
    get_histogram_boxplot(
        df,
        df_column_name,
        df.target,
    )

### Категориальные признаки

In [ ]:
# alternative
df.groupby(by=["property"])["target"].value_counts(normalize=True)

Объявим необходимые функции для построения столбчатых и круговых диаграмм

In [ ]:
def get_bar(df, df_column_name, df_target_column_name):
    fig = px.bar(
        df,
        x=df_column_name,
        y='count',
        color=df_target_column_name,
        color_discrete_map={
            'неудовлетворен': 'Coral',
            'нейтрально': 'lightyellow',
            'удовлетворен': 'SeaGreen',
        },
    )

    fig.update_layout(
        title_text=f"Распределение {df_column_name}",
        yaxis_title_text=f"{df_column_name}",
        xaxis_title_text="Кол-во",
        barmode="group",
        legend_title=None,
    )

    fig.show()

In [ ]:
def get_pie(df, df_column_name, df_target_column_name):
    unique_values_count = df[df_column_name].nunique()
    unique_values_list = list(df[df_column_name].unique())

    pie_specs = [{"type": "pie"}] * unique_values_count

    # Создание сетки для диаграмм
    fig = make_subplots(
        rows=1,
        cols=unique_values_count,
        specs=[pie_specs],
        subplot_titles=unique_values_list,
    )

    for col_num in range(1, unique_values_count + 1):
        filter = df[df_column_name] == unique_values_list[col_num - 1]
        filtered_df = df[filter]

        # Добавление диаграммы
        fig.add_trace(
            go.Pie(
                labels=filtered_df[df_target_column_name], 
                values=filtered_df["count"],
                marker=dict(colors=['lightyellow', 'Coral', 'SeaGreen']),
            ),
            row=1, col=col_num
        )
    
    fig.update_layout(
        title_text=f"Распределение {df_column_name}",
    )

    fig.show()

In [ ]:
def get_bar_pie(df, df_column_name, df_target_column_name):
    get_bar(df, df_column_name, df_target_column_name)
    get_pie(df, df_column_name, df_target_column_name)

Построим диаграммы для категориальных признаков в разрезе целевого признака

In [ ]:
for df_column_name in df.select_dtypes(exclude="number").drop(columns=["target_property"], axis=1).columns:
    df_grouped = df.groupby(by=["property", df_column_name])['property'].count().reset_index(name="example")
    df.rename(columns={'id': 'count'}, inplace=True)

    get_bar_pie(df, df_column_name, "target_property")

### Промежуточные выводы

## Сравнение тренировочных и тестовых данных

In [ ]:
# Получение описательной статистики
train_data = train_df.drop(columns=["property"], axis=1)
train_stats = train_data.describe()
print("Train:")
display(train_stats.T)

test_data = test_df.drop("property", axis=1)
test_stats = test_data.describe()
print("Test:")
display(test_stats.T)

Статистика по тренировочным и тестовым данным выглядит похожей

Сравним распределения значений

In [ ]:
train_num_data_columns = train_data.select_dtypes(include="number").columns.tolist()
unique_num_values_count = len(train_num_data_columns)

# Создание сетки для диаграмм
fig = make_subplots(
    rows=1,
    cols=unique_num_values_count,
    subplot_titles=train_num_data_columns,
)

# Визуальное сравнение распределений
for col_num in range(1, unique_num_values_count + 1):

    fig.add_trace(
        go.Histogram(
            name=f"train {train_num_data_columns[col_num - 1]}",
            x=train_data[train_num_data_columns[col_num - 1]],
            marker_color="SeaGreen"
        ),
        row=1,
        col=col_num,
    )

    fig.add_trace(
        go.Histogram(
            name=f"test {train_num_data_columns[col_num - 1]}",
            x=test_data[train_num_data_columns[col_num - 1]],
            marker_color="Coral"
        ),
        row=1,
        col=col_num,
    )

    fig.update_layout(
        barmode="group",
    )

fig.show()

Распределение значений по количественным признакам выглядит похоже

Сравним категориальные признаки

In [ ]:
train_cat_data_columns = train_data.drop("job_satisfaction_cat", axis=1).select_dtypes(exclude="number").columns.tolist()
unique_cat_values_count = len(train_cat_data_columns)

# Создание сетки для диаграмм
fig = make_subplots(
    rows=1,
    cols=unique_cat_values_count,
    subplot_titles=train_cat_data_columns,
)

# Визуальное сравнение распределений
for col_num in range(1, unique_cat_values_count + 1):
    
    fig.add_trace(
        go.Bar(
            name=f"train {train_cat_data_columns[col_num - 1]}",
            x=train_data[train_cat_data_columns[col_num - 1]].value_counts().index,
            y=train_data[train_cat_data_columns[col_num - 1]].value_counts(normalize=True),
            marker_color="SeaGreen",
        ),
        row=1,
        col=col_num,
    )

    fig.add_trace(
        go.Bar(
            name=f"test {train_cat_data_columns[col_num - 1]}",
            x=train_data[train_cat_data_columns[col_num - 1]].value_counts().index,
            y=test_data[train_cat_data_columns[col_num - 1]].value_counts(normalize=True),
            marker_color="Coral",
        ),
        row=1,
        col=col_num,
    )

    fig.update_layout(barmode='group')
fig.show()

Распределение значений по категориальным признакам тоже выглядит похоже

### Промежуточные выводы

## Корреляционный анализ

In [ ]:
# выделим колонки с количественными и категориальными данными
num_columns = list(df.select_dtypes(include="number").columns)

# построим матрицу корреляции
correlation_matrix = df.phik_matrix(interval_cols=num_columns , verbose=False)

plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f');

Мультиколлинеарности признаков (с коэффициентом зависимости больше 0.9) не наблюдается.

Судя по матрице корреляции, целевой признак (`target`) слабо зависит от следующих признаков (коэффициент зависимости меньше либо равен 0.3):
- property
- property

Умеренно либо сильно зависит от:
- property
- property

### Промежуточные выводы

Матрица корреляции показала заметную или сильную зависимость целевого признака (увольнение сотрудника из компании) от количества лет в компании, зарплаты и уровня удовлетворенности. В меньшей степени от уровня занимаемой позиции.

Это дополнительно подтверждает выводы, сделанные на предыдущих шагах анализа.

## Подготовка данных к обучению моделей

Подготовим тестовые данные, разобъем на выборки по признакам

In [ ]:
X_test_quit = test_quit_reindex_full_df.drop("quit", axis=1)
display(X_test_quit.head())

y_test_quit = test_quit_reindex_full_df["quit"]
display(y_test_quit.head())

Подготовим тренировочные данные, рассчитаем признак `job_satisfaction_rate` с помощью модели из задания 1

In [ ]:
train_quit_reindex_df = train_quit_df.set_index("id")

X_train_quit = train_quit_reindex_df.drop("quit", axis=1)
display(X_train_quit.head())

y_train_quit = train_quit_reindex_df["quit"]
display(y_train_quit.head())

In [ ]:
X_train_quit_transformed = preprocessor.transform(X_train_quit.copy())
X_train_quit["job_satisfaction_rate"] = model.predict(X_train_quit_transformed)
X_train_quit.head()

In [ ]:
X_train_quit.shape, X_test_quit.shape, y_train_quit.shape, y_test_quit.shape

Данные готовы к обучению моделей

## Обучение моделей

Подготовим функцию для создания пайплайна и обучения моделей

In [ ]:
def create_pipe_get_random_search(ohe_columns, ord_columns, num_columns):
    # Подготовим пайплайны
    # создаём пайплайн для подготовки признаков из списка ohe_columns: заполнение пропусков и OHE-кодирование
    # SimpleImputer + OHE
    ohe_pipe = Pipeline(
        [
            (
                "simpleImputer_ohe",
                missing_values_pipe,
            ),
#             (
#                 "simpleImputer_space_ohe",
#                 missing_space_values_pipe,
#             ),
            (
                "ohe",
                OneHotEncoder(
                    drop="first", 
                    handle_unknown="ignore", 
                    sparse_output=False
                ),
            ),
        ]
    )

    # cоздаём пайплайн для подготовки признаков из списка ord_columns: заполнение пропусков и Ordinal-кодирование
    # SimpleImputer + OE
    ord_pipe = Pipeline(
        [
            (
                "simpleImputer_before_ord",
                missing_values_pipe,
            ),
            (
                "ord",
                OrdinalEncoder(
                    categories=[
                        ["junior", "middle", "senior"],
                        ["low", "medium", "high"],
                    ],
                    handle_unknown="use_encoded_value",
                    unknown_value=np.nan,
                ),
            ),
            (
                "simpleImputer_after_ord",
                missing_values_pipe,
            ),
        ]
    )

    # пайплайн для подготовки данных
    data_preprocessor = ColumnTransformer(
        [
            ("ohe", ohe_pipe, ohe_columns),
            ("ord", ord_pipe, ord_columns),
            ("num", MinMaxScaler(), num_columns),
        ],
        remainder="passthrough",
    )

    # итоговый пайплайн: подготовка данных и модель
    pipe_final = Pipeline(
        [
            ("preprocessor", data_preprocessor),
            ("models", DecisionTreeClassifier(random_state=RANDOM_STATE)),
        ]
    )

    # Подготовим словари для моделей данных
    param_grid = [
        # словарь для модели DecisionTreeClassifier()
        {
            "models": [DecisionTreeClassifier(random_state=RANDOM_STATE)],
            "models__max_depth": range(2, 20),
            "models__max_features": range(2, 20),
            "preprocessor__num": [StandardScaler(), MinMaxScaler(), RobustScaler(), "passthrough"],
        },
        # словарь для модели LogisticRegression()
        {
            "models": [
                LogisticRegression(
                    random_state=RANDOM_STATE,
                    solver="liblinear"
                )
            ],
            "models__C": range(1, 5),
            "preprocessor__num": [StandardScaler(), MinMaxScaler(), RobustScaler(), "passthrough"],
        },
        # словарь для модели SVC()
        {
            "models": [
                SVC(random_state=RANDOM_STATE, kernel="linear", probability=True)
            ],
            "models__C": range(2, 5),
            "preprocessor__num": [StandardScaler(), MinMaxScaler(), RobustScaler(), "passthrough"],
        },
    ]

    randomized_search = RandomizedSearchCV(
        pipe_final,
        param_grid,
        n_iter=40,
        cv=5,
        scoring="roc_auc",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    return randomized_search

Разобьем признаки на категориальные и количественные. Для OneHotEncoder и OrdinalEncoder.

In [ ]:
RANDOM_STATE = 1

cat_columns = list(X_train_quit.select_dtypes(exclude="number").columns)
print("cat_columns:", cat_columns)

num_columns = list(X_train_quit.select_dtypes(include="number").columns)
print("num_columns:", num_columns)

ohe_columns = [
    "dept",
    "last_year_promo",
    "last_year_violations",
]
print("ohe_columns:", ohe_columns)

ord_columns = [
    "level",
    "workload",
]
print("ord_columns:", ord_columns)

In [ ]:
randomized_search = create_pipe_get_random_search(ohe_columns, ord_columns, num_columns)
randomized_search.fit(X_train_quit, y_train_quit)

print('Лучшая модель и её параметры:\n\n', randomized_search.best_estimator_)
print('Метрика лучшей модели на кросс-валидации:', randomized_search.best_score_)

# проверим работу модели на тестовой выборке
y_test_pred = randomized_search.predict_proba(X_test_quit)[:,1]
print(f'Метрика ROC_AUC_SCORE на тестовой выборке: {roc_auc_score(y_test_quit, y_test_pred)}')

ROC_AUC_SCORE на тестовой выборке составляет 0.93, что соответствует требованию в задании.

Получим топ-10 моделей для анализа

In [ ]:
search_results_df = pd.DataFrame(randomized_search.cv_results_)

# Топ-10 лучших результатов
top_10_results = search_results_df.sort_values(by='rank_test_score').head(10)

# Вывод основных метрик
print("Топ-10 лучших комбинаций параметров:")
top_10_results

Лучшая модель показала стабильные результаты на кросс-валидации (std_test_score = 0.01).

### Промежуточные выводы

- Удалили дубли в тренировочных данных
- Разбили признаки на категориальные и количественные. Категориальные признаки в свою очередь разбили для использования разными энкодерами, OneHotEncoder и OrdinalEncoder
- Использовали ROC-AUC метрику для оценки моделей
- Составили пайплайн, заполнили пропуски в данных и нашли лучшую модель и ее гиперпараметры для задачи регрессии

Лучшей признана модель регрессионного дерева решений со следующими параметрами
DecisionTreeClassifier(max_depth=5, max_features=11, random_state=1)

ROC-AUC показала значение 0.93 на тренировочной выборке и значение 0.93 на тестовой. Значение на тестовой выборке соответствует требованиям в задаче

## Анализ важности признаков

In [ ]:
preprocessor = randomized_search.best_estimator_.named_steps['preprocessor']
model = randomized_search.best_estimator_.named_steps['models']

In [ ]:
X_train_quit_piped = preprocessor.fit_transform(X_train_quit.copy())
feature_names = preprocessor.get_feature_names_out()

# Получение важности признаков
model.fit(X_train_quit_piped, y_train_quit)
feature_importances = model.feature_importances_

# Вывод результатов
feature_importances_df = pd.DataFrame(data=zip(feature_names, feature_importances), columns=["feature", "importance"])
feature_importances_df.sort_values(by="importance", ascending=False)

Построим SHAP диаграммы для оценки влияния признаков на целевой

In [ ]:
original_features = list(X_train_job_rate.columns)

X_test_piped = preprocessor.transform(X_test_job_rate.copy())
explainer = shap.Explainer(model, X_test_piped)
feature_names = preprocessor.get_feature_names_out()
shap_values = explainer(X_test_piped, check_additivity=False)
 
shap.summary_plot(shap_values, X_test_piped, feature_names=feature_names, max_display=15, plot_size=(15, 8))

In [ ]:
shap.summary_plot(shap_values, X_test_piped, plot_type="bar", feature_names=feature_names, max_display=15, plot_size=(15, 8))

### Промежуточные выводы

На основании анализа важности признаков для модели и SHAP диаграмм видно, что на увольнение сотрудников из компании влияют следующие признаки:
- property
- property

Эти данные подтверждают выводы, сделанные на шаге Исследовательский анализ данных. Несмотря на то, что качество модели соответствует заявленным требованиям, ее качество можно попробовать улучшить, если удалить из датасета колонки, важность которых составляет 0.

## Общий вывод